# 📄 LLM-Based Workflows - Invoice Processing Pipeline

**Author:** *José Ignacio Orlando, PhD*

In this example, we're going to learn how to build a complete invoice processing workflow using LangGraph and LLMs. This is a practical, real-world application that demonstrates how to structure complex data extraction and processing pipelines using this library, and integrating LLMs in the process.

Our goal is to create an automated system that can:
- Extract text from PDF invoices
- Use LLMs to parse and structure the invoice data
- Normalize and validate the extracted information
- Export the results to CSV for further analysis

This workflow showcases the power of combining LangGraph's orchestration capabilities with LLM-based data extraction, creating a robust solution for document processing tasks.

## Step 1 - Environment Setup

Before we can start processing invoices, we need to set up our environment properly. This involves:

1. **Loading environment variables**: We use `python-dotenv` to securely load our OpenAI API key from a `.env` file
2. **API key validation**: We ensure the required API key is present before proceeding
3. **Model configuration**: We set up the OpenAI model we'll use for structured data extraction

The `load_dotenv()` function reads the `.env` file in your project directory, where you should store your OpenAI API key as `OPENAI_API_KEY=your_key_here`. This is a security best practice that prevents accidentally committing API keys to version control.

**Important**: Always remember to **not include .env files in your repository**. They should be part of your `.gitignore` file to prevent leaking your API keys online.

In [1]:
from os import getenv
from dotenv import load_dotenv

load_dotenv()
assert getenv("OPENAI_API_KEY") is not None, "Please set OPENAI_API_KEY in .env or environment."

OPENAI_MODEL = getenv("OPENAI_MODEL", "gpt-5")

## Step 2 - Define Data Models

Now we need to define the structure of our invoice data using Pydantic models. This is crucial for ensuring data consistency and validation throughout our pipeline.

We define two main models:
- **`LineItem`**: Represents individual items on an invoice with description, quantity, unit price, and total
- **`InvoiceSchema`**: The complete invoice structure including supplier information, dates, totals, and line items

Using Pydantic models provides several benefits:
- **Type safety**: Ensures our data has the correct types
- **Validation**: Automatically validates data as it flows through the pipeline
- **Documentation**: The models serve as clear documentation of our data structure
- **LLM integration**: Works seamlessly with LangChain's structured output features

Notice how we use optional fields (`str | None`) for information that might not always be present in invoices, and we include field validators to ensure data quality.

In [2]:
from typing import List
from pydantic import BaseModel, Field, field_validator

class LineItem(BaseModel):
    description: str = Field(description="item description")
    quantity: float = Field(description="quantity purchased")
    unit_price: float = Field(description="price per unit")
    total: float = Field(description="line item total")
    
class InvoiceSchema(BaseModel):
    invoice_number: str
    invoice_date: str   # normalized later to YYYY-MM-DD when possible
    supplier_name: str
    
    supplier_address: str | None = None       # optional (may be omitted) and nullable
    supplier_vat: str | None = None           # optional + nullable
    customer_name: str | None = None          # optional + nullable
    customer_address: str | None = None       # optional + nullable
    currency: str | None = Field(default=None, description="ISO 4217 (e.g., USD, EUR, ARS)")
    subtotal: float | None = None
    tax: float | None = None

    total: float                              # required
    line_items: List[LineItem]                # required

    @field_validator("invoice_date")
    @classmethod
    def nonempty(cls, v):
        if not v or not v.strip():
            raise ValueError("invoice_date required")
        return v


## Step 3 - Define the Graph State

As in all our LangGraph examples, we need to define the state that will flow through our processing pipeline. The `GraphState` serves as our shared memory, allowing different nodes to access and update information as the invoice is processed.

Our state tracks the complete processing pipeline:
- **`pdf_path`**: The input PDF file path
- **`raw_text`**: Extracted text from the PDF
- **`extracted`**: Structured data extracted by the LLM
- **`normalized`**: Cleaned and validated invoice data
- **`csv_path`**: Output CSV file path

This state design allows us to track the invoice through each processing stage, making it easy to debug issues and understand the data transformation pipeline.

In [3]:
from typing import TypedDict

class GraphState(TypedDict):
    pdf_path: str
    raw_text: str | None
    extracted: InvoiceSchema | None
    normalized: InvoiceSchema | None
    csv_path: str | None

## Step 4 - Set up the LLM for Structured Output

Here we configure our LLM to return structured data that matches our Pydantic models. This is a powerful feature that allows us to get consistent, validated data from the LLM.

The key components are:
- **`ChatOpenAI`**: Our interface to OpenAI's models
- **`with_structured_output()`**: This method configures the LLM to return data in our specified schema format

Under the hood, LangChain uses function calling (tool calling) to ensure the LLM returns data in the exact format we need. This eliminates the need for manual parsing and validation of LLM responses, making our pipeline much more reliable.

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=OPENAI_MODEL)

# Wrap it to return structured data parsed into our Pydantic model
# (LangChain's with_structured_output uses tool/function-calling under the hood)
structured_llm = llm.with_structured_output(InvoiceSchema)

## Step 5 - PDF Text Extraction

Our first processing node handles the extraction of text from PDF files. This is a critical step because the quality of our extraction directly impacts the accuracy of our LLM-based parsing.

The `load_pdf` function:
- Opens the PDF file using `pdfplumber`
- Extracts text from each page
- Combines all text into a single string
- Stores the result in our graph state

**Note**: This implementation works with text-based PDFs. For scanned PDFs, you would need to add OCR (Optical Character Recognition) upstream to convert images to text first.

The extracted text will serve as input to our LLM, so it's important that it's clean and complete.

In [5]:
import pdfplumber

# Node 1: Load & extract text from PDF (text PDFs; add OCR upstream for scanned PDFs)
def load_pdf(state: GraphState) -> GraphState:
    pdf_path = state["pdf_path"]
    text_chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            text_chunks.append(text)
    state["raw_text"] = "\n".join(text_chunks).strip()
    return state




## Step 6 - LLM-Based Data Extraction

This is where the magic happens! Our `extract_with_llm` node uses the power of LLMs to intelligently parse the raw invoice text and extract structured data.

The process involves:
1. **System prompt**: We provide clear instructions about the extraction task
2. **User prompt**: We pass the raw invoice text to the LLM
3. **Structured output**: The LLM returns data in our exact `InvoiceSchema` format
4. **State update**: We store the extracted data in our graph state

The LLM is particularly good at this task because it can:
- Understand different invoice formats and layouts
- Handle variations in terminology and formatting
- Extract information even when it's not perfectly structured
- Make intelligent guesses about missing information

This demonstrates the power of combining traditional data processing with AI capabilities.

In [6]:
# Node 2: LLM extraction (ChatOpenAI + structured output)
def extract_with_llm(state: GraphState) -> GraphState:
    raw = state.get("raw_text") or ""
    system_prompt = (
        "You are an expert data extractor for invoices. "
        "Given raw invoice text, produce a structured object that adheres to the provided schema. "
        "If a value is missing, omit it or use null when allowed. Do not fabricate fields."
    )
    user = f"Invoice text:\n\n{raw}\n\nReturn only the structured object."

    # LangChain accepts lists of messages or strings; we’ll use messages for clarity
    result: InvoiceSchema = structured_llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user},
        ]
    )
    state["extracted"] = result
    return state

## Step 7 - Data Normalization and Validation

After extracting the raw data, we need to clean and normalize it to ensure consistency. The `normalize` function handles several important tasks:

- **Date normalization**: Converts various date formats to ISO standard (YYYY-MM-DD)
- **Currency handling**: Sets default currency if none is specified
- **Data validation**: Ensures all required fields are present
- **Format consistency**: Standardizes data formats across different invoices

The normalization process includes intelligent date parsing that can handle common formats like DD/MM/YYYY and MM/DD/YYYY, automatically converting them to the ISO standard.

This step is crucial for downstream processing and analysis, as it ensures all our invoice data follows consistent formats and standards.

In [7]:
import re
from dataclasses import dataclass


# Node 3: Normalize & light validation
@dataclass
class NormalizationConfig:
    default_currency: str = "USD"

def _guess_date_iso(d: str) -> str:
    d = d.strip()
    # Already ISO
    if re.match(r"^\d{4}-\d{2}-\d{2}$", d):
        return d
    # DD/MM/YYYY or MM/DD/YYYY -> assume D/M/Y, reorder
    m = re.match(r"^(\d{1,2})[./-](\d{1,2})[./-](\d{4})$", d)
    if m:
        a, b, y = m.groups()
        day, month = int(a), int(b)
        return f"{y}-{month:02d}-{day:02d}"
    return d  # leave unchanged if unknown

def normalize(state: GraphState, cfg: NormalizationConfig = NormalizationConfig()) -> GraphState:
    inv: InvoiceSchema = state["extracted"]
    currency = inv.currency or cfg.default_currency
    date_iso = _guess_date_iso(inv.invoice_date)

    normalized = InvoiceSchema(
        invoice_number=inv.invoice_number,
        invoice_date=date_iso,
        supplier_name=inv.supplier_name,
        supplier_address=inv.supplier_address,
        supplier_vat=inv.supplier_vat,
        customer_name=inv.customer_name,
        customer_address=inv.customer_address,
        currency=currency,
        subtotal=inv.subtotal,
        tax=inv.tax,
        total=inv.total,
        line_items=inv.line_items,
    )
    state["normalized"] = normalized
    return state

## Step 8 - CSV Export

The final step in our pipeline exports the processed invoice data to a CSV file. This makes the data accessible for further analysis, reporting, or integration with other systems.

The `write_csv` function:
- **Creates headers**: Defines the CSV column structure
- **Handles first write**: Creates the file with headers if it doesn't exist
- **Appends data**: Adds new invoice records to existing files
- **JSON serialization**: Stores line items as JSON strings for complex data

The CSV format is particularly useful because:
- It's widely supported by spreadsheet applications
- It's easy to import into databases
- It's human-readable for manual inspection
- It supports incremental updates (appending new invoices)

This completes our invoice processing pipeline, transforming unstructured PDF data into structured, analyzable information.

In [8]:
import csv
from os import path


# Node 4: Write/append to CSV (flat header + items JSON column)
CSV_HEADERS = [
    "invoice_number","invoice_date","supplier_name","supplier_address","supplier_vat",
    "customer_name","customer_address","currency","subtotal","tax","total","line_items_json"
]

def write_csv(state: GraphState) -> GraphState:
    inv: InvoiceSchema = state["normalized"]
    csv_path = state.get("csv_path") or "invoices.csv"
    first_write = not path.exists(csv_path)

    row = {
        "invoice_number": inv.invoice_number,
        "invoice_date": inv.invoice_date,
        "supplier_name": inv.supplier_name,
        "supplier_address": inv.supplier_address or "",
        "supplier_vat": inv.supplier_vat or "",
        "customer_name": inv.customer_name or "",
        "customer_address": inv.customer_address or "",
        "currency": inv.currency or "",
        "subtotal": inv.subtotal if inv.subtotal is not None else "",
        "tax": inv.tax if inv.tax is not None else "",
        "total": inv.total,
        "line_items_json": [li.model_dump() for li in inv.line_items],
    }

    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_HEADERS)
        if first_write:
            writer.writeheader()
        writer.writerow({**row, "line_items_json": str(row["line_items_json"])})

    state["csv_path"] = csv_path
    return state

## Step 9 - Build the LangGraph Workflow

Now we put everything together to create our complete invoice processing workflow. This is where LangGraph's orchestration capabilities shine, managing the flow of data through our processing pipeline.

Our workflow follows a linear sequence:
1. **START** → `load_pdf` (extract text from PDF)
2. `load_pdf` → `extract_with_llm` (parse with LLM)
3. `extract_with_llm` → `normalize` (clean and validate data)
4. `normalize` → `write_csv` (export to CSV)
5. `write_csv` → **END** (complete processing)

Each node in our graph has a specific responsibility, and the state flows through them in sequence. This design makes it easy to:
- Debug individual steps
- Add new processing stages
- Handle errors at specific points
- Monitor the processing pipeline

The `compile()` method finalizes our workflow and makes it ready for execution.

In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables import RunnableConfig

# %%
workflow = StateGraph(GraphState)

# Register nodes
workflow.add_node("load_pdf", load_pdf)
workflow.add_node("extract_with_llm", extract_with_llm)
workflow.add_node("normalize", normalize)
workflow.add_node("write_csv", write_csv)

# Linear chain: START → load_pdf → extract_with_llm → normalize → write_csv → END
workflow.add_edge(START, "load_pdf")
workflow.add_edge("load_pdf", "extract_with_llm")
workflow.add_edge("extract_with_llm", "normalize")
workflow.add_edge("normalize", "write_csv")
workflow.add_edge("write_csv", END)

app = workflow.compile()


To ease the execution, we're gonna implement an auxiliary function that will take the PDF path of an invoice and a CSV path, and will create the graph state and execute the graph once.

In [10]:
# %%
def run_once(pdf_path: str, csv_path: str = "invoices.csv"):
    state: GraphState = {
        "pdf_path": pdf_path,
        "raw_text": None,
        "extracted": None,
        "normalized": None,
        "csv_path": csv_path,
    }
    result = app.invoke(state, config=RunnableConfig(run_name="invoice_to_csv"))
    print(f"Done. Appended to: {result['csv_path']}")
    return result


## Step 11 - Test with a Single Invoice

Let's test our pipeline with a single invoice to make sure everything works correctly. This is a crucial step before processing multiple invoices.

**What just happened?**
Our pipeline successfully:
1. Extracted text from the PDF invoice
2. Used the LLM to parse and structure the data
3. Normalized the extracted information
4. Exported the results to CSV

The processing completed successfully, and we can see the confirmation message indicating the data was appended to our CSV file. This demonstrates that our complete pipeline is working as expected.

In [12]:
# %%
# Replace with a local path to a sample invoice you downloaded.
pdf_example = "./samples/demo-invoice-20tax-1.pdf"   # <- update this path
_ = run_once(pdf_example, "invoices.csv")


Done. Appended to: invoices.csv


## Step 12 - Batch Processing Multiple Invoices

Now let's scale up and process multiple invoices in batch. This demonstrates how our pipeline can handle real-world scenarios where you need to process many documents.

**What just happened?**
We successfully processed 10 invoices in sequence, with each one:
- Going through the complete extraction pipeline
- Being parsed by the LLM
- Having its data normalized and validated
- Being exported to the same CSV file

The batch processing shows the scalability of our approach - we can easily process hundreds or thousands of invoices by simply adding more files to our processing loop.

**Next Steps / How to Extend**

This invoice processing pipeline provides a solid foundation that you can extend in many ways:

- **Error handling**: Add try-catch blocks to handle PDF parsing errors or LLM failures
- **Parallel processing**: Use LangGraph's parallel execution capabilities to process multiple invoices simultaneously
- **Data validation**: Add more sophisticated validation rules for specific invoice types
- **OCR integration**: Add OCR capabilities for scanned PDFs
- **Database integration**: Store results in a database instead of CSV
- **Web interface**: Create a web application for uploading and processing invoices
- **Monitoring**: Add logging and monitoring for production deployments
- **Custom schemas**: Create different schemas for different types of documents (receipts, contracts, etc.)

The modular design of our pipeline makes it easy to add these enhancements without disrupting the core functionality.

In [14]:
for i in range(1,11):
    print(f"Processing invoice #{i}/10")
    _ = run_once(f"./samples/demo-invoice-20tax-{  i}.pdf", "invoices.csv")

Processing invoice #1/10
Done. Appended to: invoices.csv
Processing invoice #2/10
Done. Appended to: invoices.csv
Processing invoice #3/10
Done. Appended to: invoices.csv
Processing invoice #4/10
Done. Appended to: invoices.csv
Processing invoice #5/10
Done. Appended to: invoices.csv
Processing invoice #6/10
Done. Appended to: invoices.csv
Processing invoice #7/10
Done. Appended to: invoices.csv
Processing invoice #8/10
Done. Appended to: invoices.csv
Processing invoice #9/10
Done. Appended to: invoices.csv
Processing invoice #10/10
Done. Appended to: invoices.csv
